# 3 · Fine-tune the image model on YOUR labelled images

The pretrained model scored ROC-AUC 0.58 on your data (near a coin flip). This notebook **trains it further on your own real/fake images**, then tests it on images it never saw.

**Before you start**
* Runtime → Change runtime type → **T4 GPU**.
* Put **all** your images in two folders whose names contain *real* and *fake* (e.g. `REAL IMAGE`, `FAKE IMAGE`) and zip the parent folder. About 2,000 per class is plenty.

**What you get at the end:** a held-out test accuracy, a `image_finetuned.zip` model folder, and a check that the project's own detector loads it.

Honest limit: the model will be good at *the kind of fake in your dataset*. It can be much worse on other generators or on phone photos. Say that when you present.

## 1. Install

In [ ]:
!pip install -q transformers accelerate pillow numpy
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

## 2. Load the project code (upload `deepfake_detection_suite_FINAL.zip`)

In [ ]:
import os, glob, zipfile
from google.colab import files
up = files.upload()
with zipfile.ZipFile(list(up.keys())[0]) as z:
    z.extractall("/content/project")
hits = glob.glob("/content/project/**/06_dataset_and_evaluation/batch_evaluate.py", recursive=True)
assert hits, "batch_evaluate.py not found - did you upload the FINAL project zip?"
SUITE_ROOT = os.path.dirname(os.path.dirname(hits[0]))
print("Suite root:", SUITE_ROOT)

## 3. Load ALL your labelled images

In [ ]:
import hashlib
USE_DRIVE = False
DRIVE_FOLDER = "/content/drive/MyDrive/DATASET"      # only if USE_DRIVE = True

IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive"); SRC = DRIVE_FOLDER
else:
    up = files.upload()
    with zipfile.ZipFile(list(up.keys())[0]) as z:
        z.extractall("/content/labelled")
    SRC = "/content/labelled"

def discover_images(base):
    data = {"real": [], "fake": []}
    for root, dirs, fs in os.walk(base):
        dirs[:] = [d for d in dirs if d != "__MACOSX"]
        parts = [] if root == base else os.path.relpath(root, base).lower().replace("\\", "/").split("/")
        label = next(("fake" if "fake" in p else "real" for p in reversed(parts) if "fake" in p or "real" in p), None)
        if label is None or any("video" in p for p in parts):
            continue
        for f in fs:
            if not f.startswith(".") and f.lower().endswith(IMG_EXT):
                data[label].append(os.path.join(root, f))
    return {k: sorted(v) for k, v in data.items()}

DATA = discover_images(SRC)
print({k: len(v) for k, v in DATA.items()})
assert min(len(v) for v in DATA.values()) >= 50, "Need at least 50 images per class (folder names must contain 'real' / 'fake')."

# Remove exact duplicate files so the same image can't sit in both train and test (that would inflate the score)
seen, dups = {}, 0
for label in DATA:
    keep = []
    for p in DATA[label]:
        h = hashlib.md5(open(p, "rb").read()).hexdigest()
        if h in seen:
            dups += 1
        else:
            seen[h] = label; keep.append(p)
    DATA[label] = keep
print("duplicates removed:", dups, "| images kept:", {k: len(v) for k, v in DATA.items()})

## 4. Split: 70% train · 15% validation · 15% test
The **test** images are never used for training or for choosing the best epoch. They give the honest number.

In [ ]:
import random, numpy as np
SEED = 42
split = {"train": [], "val": [], "test": []}
for label, y in (("real", 0), ("fake", 1)):
    fl = DATA[label][:]
    random.Random(SEED).shuffle(fl)
    n = len(fl); a, b = int(n * 0.70), int(n * 0.85)
    split["train"] += [(p, y) for p in fl[:a]]
    split["val"]   += [(p, y) for p in fl[a:b]]
    split["test"]  += [(p, y) for p in fl[b:]]
for k, v in split.items():
    print(f"{k:5s}: {len(v):5d}  (fake = {sum(y for _, y in v)})")

## 5. Settings
Defaults suit ~4,000 images on a T4 (about 10-15 minutes in total).

In [ ]:
BASE   = "prithivMLmods/Deep-Fake-Detector-v2-Model"   # start from the existing detector
EPOCHS = 4
BATCH  = 32
LR     = 2e-5
OUT    = "/content/image_finetuned"

## 6. Train

In [ ]:
import json, time, copy
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForImageClassification, AutoImageProcessor, get_linear_schedule_with_warmup

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
processor = AutoImageProcessor.from_pretrained(BASE)      # the SAME preprocessing the detector uses at inference

class DS(Dataset):
    def __init__(self, items, train): self.items, self.train = items, train
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        img = Image.open(p).convert("RGB")
        if self.train and random.random() < 0.5:
            img = ImageOps.mirror(img)
        return processor(images=img, return_tensors="pt")["pixel_values"][0], y

def loader(items, train):
    return DataLoader(DS(items, train), batch_size=BATCH, shuffle=train, num_workers=2, pin_memory=(device == "cuda"))

def auc_score(scores, labels):
    s, y = np.asarray(scores, float), np.asarray(labels, bool)
    p, n = s[y], s[~y]
    return float(((p[:, None] > n[None, :]).sum() + 0.5 * (p[:, None] == n[None, :]).sum()) / (len(p) * len(n)))

@torch.no_grad()
def predict(model, dl):
    model.eval(); probs, ys, loss_sum = [], [], 0.0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(pixel_values=x).logits
        loss_sum += torch.nn.functional.cross_entropy(logits.float(), y, reduction="sum").item()
        probs += torch.softmax(logits.float(), dim=-1)[:, 1].cpu().tolist(); ys += y.cpu().tolist()
    probs, ys = np.array(probs), np.array(ys)
    return {"acc": float(((probs >= 0.5) == ys.astype(bool)).mean() * 100), "auc": auc_score(probs, ys),
            "loss": loss_sum / len(ys), "probs": probs, "ys": ys}

model = AutoModelForImageClassification.from_pretrained(
    BASE, num_labels=2, id2label={0: "Real", 1: "Fake"}, label2id={"Real": 0, "Fake": 1}, ignore_mismatched_sizes=True).to(device)

dl_train, dl_val, dl_test = loader(split["train"], True), loader(split["val"], False), loader(split["test"], False)

# Reference point: what does the model score on the validation images BEFORE any training?
# (its head is re-initialised only if shapes differ; the label order is set above, so treat this as a rough baseline)
r0 = predict(model, dl_val)
print(f"before fine-tuning  | val acc {r0['acc']:5.1f}%  AUC {r0['auc']:.3f}")

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * len(dl_train)
sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

best, best_state, history = (-1, -1), None, []
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    model.train(); tl, n = 0.0, 0
    for x, y in dl_train:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            loss = torch.nn.functional.cross_entropy(model(pixel_values=x).logits.float(), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item() * len(y); n += len(y)
    r = predict(model, dl_val)
    history.append({"epoch": ep, "train_loss": tl / n, "val_loss": r["loss"], "val_acc": r["acc"], "val_auc": r["auc"]})
    print(f"epoch {ep}/{EPOCHS} | train loss {tl/n:.4f} | val loss {r['loss']:.4f} | val acc {r['acc']:5.1f}% | val AUC {r['auc']:.3f} | {time.time()-t0:.0f}s")
    if (r["auc"], r["acc"]) > best:
        best, best_state = (r["auc"], r["acc"]), copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
model.load_state_dict(best_state)
print("Best epoch restored (chosen on the validation set only).")

## 7. The honest number: score on the TEST images
These images were never used for training or for picking the epoch. Run this cell once and report what it prints.

In [ ]:
rt = predict(model, dl_test)
pred, ys, p = rt["probs"] >= 0.5, rt["ys"].astype(bool), rt["probs"]
tp, tn = int((pred & ys).sum()), int((~pred & ~ys).sum()); fp, fn = int((pred & ~ys).sum()), int((~pred & ys).sum())
lead = np.abs(2 * p - 1) * 100                      # decision margin in points, as in the app
decided = lead > 6                                  # the app's UNCERTAIN band
print(f"TEST images: {len(ys)}")
print(f"  accuracy          : {rt['acc']:.1f}%")
print(f"  ROC-AUC           : {rt['auc']:.3f}   (was 0.58 before fine-tuning)")
print(f"  confusion         : real->real {tn} | real->fake {fp} | fake->fake {tp} | fake->real {fn}")
print(f"  answered (|margin|>6): {decided.mean()*100:.1f}%  accuracy on those: {((pred==ys)[decided]).mean()*100:.1f}%")
print(f"  very confident (margin>=80): {(lead>=80).mean()*100:.1f}% of images, right {((pred==ys)[lead>=80]).mean()*100:.1f}% of the time")

## 8. Save the model (plus a report with the test accuracy)

In [ ]:
import shutil
os.makedirs(OUT, exist_ok=True)
model.cpu().save_pretrained(OUT); processor.save_pretrained(OUT)
report = {"base_model": BASE, "date": time.strftime("%Y-%m-%d"), "epochs": EPOCHS, "lr": LR, "seed": SEED,
          "n_train": len(split["train"]), "n_val": len(split["val"]), "n_test": len(split["test"]),
          "test_accuracy_pct": round(rt["acc"], 2), "test_auc": round(rt["auc"], 4),
          "test_confusion": {"tn": tn, "fp": fp, "fn": fn, "tp": tp}, "history": history,
          "test_files": sorted(os.path.basename(pth) for pth, _ in split["test"])}
json.dump(report, open(os.path.join(OUT, "training_report.json"), "w"), indent=1)
shutil.make_archive("/content/image_finetuned", "zip", "/content", "image_finetuned")
print("Saved:", OUT, "| zip size: %.0f MB" % (os.path.getsize("/content/image_finetuned.zip") / 1e6))

## 9. Check that the PROJECT's detector loads and uses the new model
This copies the model into `models/image_finetuned/` inside the project (where the detector looks for it) and runs a few test images through the real pipeline.

In [ ]:
import sys
dest = os.path.join(SUITE_ROOT, "models", "image_finetuned")
shutil.rmtree(dest, ignore_errors=True); shutil.copytree(OUT, dest)
sys.path[:0] = [f"{SUITE_ROOT}/01_image_detection", SUITE_ROOT]
from image_detector import ImageDeepfakeDetector
det = ImageDeepfakeDetector()
print("Detector is using:", det.model_id, "| ML loaded:", det._ml_available)
assert det._ml_available and "image_finetuned" in det.model_id, "The detector did not pick up the fine-tuned model"
ok = 0; sample = random.Random(1).sample(split["test"], 12)
for pth, y in sample:
    r = det.predict(pth)
    good = (r.verdict == "FAKE") == bool(y) and r.verdict != "UNCERTAIN"; ok += good
    print(f"{'fake' if y else 'real':4s} -> {r.verdict:9s} fake={r.confidence_fake_pct:6.2f}%  {'OK' if good else 'x'}")
print(f"{ok}/{len(sample)} correct in this small sample")
print(r.model_used, "|", r.model_reported_accuracy_pct, "% held-out accuracy")

## 10. Full-pipeline test on the held-out images (optional, ~5 min)
Runs the same evaluator as notebook 2, but only on the test images, so the number is honest.

In [ ]:
import subprocess
T = "/content/test_only"; shutil.rmtree(T, ignore_errors=True)
for label, y in (("real", 0), ("fake", 1)):
    os.makedirs(f"{T}/{label}", exist_ok=True)
    for i, (pth, yy) in enumerate(split["test"]):
        if yy == y: os.symlink(pth, f"{T}/{label}/{i:05d}_" + os.path.basename(pth))
os.makedirs("/content/eval_out", exist_ok=True)
cmd = [sys.executable, "-u", f"{SUITE_ROOT}/06_dataset_and_evaluation/batch_evaluate.py", "--real_dir", f"{T}/real", "--fake_dir", f"{T}/fake",
       "--save_csv", "/content/eval_out/test_results.csv", "--save_json", "/content/eval_out/test_report.json"]
p = subprocess.Popen(cmd, cwd="/content/eval_out", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout: print(line, end="")
p.wait()

## 11. Download the model
Download `image_finetuned.zip` (about 340 MB) - or save it to Google Drive, which is faster for a file this size.
Then use it in notebook 1 (section 3 loads it) or unzip it into `deepfake_detection_suite/models/` on your laptop.

In [ ]:
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy("/content/image_finetuned.zip", "/content/drive/MyDrive/image_finetuned.zip")
    print("Copied to Google Drive: MyDrive/image_finetuned.zip")
files.download("/content/image_finetuned.zip")